# 24. What is MCP (Model Context Protocol)?

**MCP (Model Context Protocol)** is a **standard way to give tools to an AI agent**.

Instead of writing a custom tool for every agent, you run a small **MCP server** that shares tools, and **any** agent can plug in and use them.

> This notebook explains the **concept** and shows **sample code**.
> The full, runnable demo is in two files next to this notebook:
> **`mcp_math_server.py`** and **`mcp_agent_demo.py`**.

## Real-life analogy

MCP is like a **USB-C port** for AI agents.

- Before USB, every device needed its **own special cable**.
- USB-C made **one standard plug** that fits everything.

MCP is that one standard plug: build a tool **once** as an MCP server, and **any** agent (CrewAI, Claude, and others) can use it.

## Why MCP?

| Without MCP | With MCP |
|-------------|----------|
| Write a new tool for every agent | Write the tool **once** |
| Tools are locked to one app | **Any** agent can use the same tool |
| Different setup each time | One **standard** way to connect |
| Hard to share | Easy to **share and reuse** |

## How it works (simple picture)

```
MCP Server  --->  shares tools (like "add")
     |
     |   (agent connects)
     v
CrewAI Agent  --->  uses those tools to do its task
```

Two parts:
1. An **MCP server** = a small program that offers tools.
2. An **agent** = connects to the server and uses those tools.

> Install MCP support once:  `pip install "crewai-tools[mcp]"`

## Sample 1: The MCP server  (`mcp_math_server.py`)

A tiny server that shares **one** tool called `add`.
Any function with **`@mcp.tool()`** becomes a tool the agent can use.

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("MathServer")

@mcp.tool()                      # this turns the function into a tool
def add(a: int, b: int) -> int:
    """Add two numbers and return the result."""
    return a + b

if __name__ == "__main__":
    mcp.run(transport="stdio")   # start the server
```

## Sample 2: Connect the server to an agent  (`mcp_agent_demo.py`)

We tell CrewAI how to **start** the server, then give its tools to an agent with `tools=mcp_tools`.
The `with` block opens the connection and closes it automatically.

```python
import sys
from dotenv import load_dotenv
load_dotenv()

from crewai import Agent, Task, Crew, LLM
from crewai_tools import MCPServerAdapter
from mcp import StdioServerParameters

# How to start our MCP server (run the server file with Python)
server_params = StdioServerParameters(command=sys.executable, args=["mcp_math_server.py"])

# Connect to the server. 'mcp_tools' are the tools it shares.
with MCPServerAdapter(server_params) as mcp_tools:
    agent = Agent(
        role="Math Helper",
        goal="Solve math using the MCP tools",
        backstory="You always use the tools from the MCP server.",
        tools=mcp_tools,                 # <-- MCP tools given to the agent
        llm=LLM(model="gpt-4o-mini"),
    )
    task = Task(
        description="What is 8 plus 5? Use your tool to add them.",
        expected_output="The number answer.",
        agent=agent,
    )
    crew = Crew(agents=[agent], tasks=[task])
    print("Answer:", crew.kickoff())     # prints: Answer: 13
```

## How to run the demo

MCP starts a **small background program (the server)**. This works perfectly when you run it
as a normal Python file, so the demo lives in **`.py`** files, not in this notebook.

In a terminal (in this `CrewAI_Framework` folder):

```
pip install "crewai-tools[mcp]"
python mcp_agent_demo.py
```

`mcp_agent_demo.py` automatically starts `mcp_math_server.py`, connects the agent to it,
and the agent uses the MCP `add` tool. Expected output:

```
Tools the agent got from MCP: ['add']
Answer: 13
```

> Tip: you need your `OPENAI_API_KEY` in a `.env` file in this folder (the agent's brain).

## Key points to remember

- **MCP (Model Context Protocol)** is a **standard way to share tools** with AI agents.
- It is like a **USB-C port**: build a tool once, and **any** agent can use it.
- An **MCP server** is a small program; each `@mcp.tool()` function becomes a tool.
- Connect it in CrewAI with **`MCPServerAdapter(server_params)`** inside a `with` block.
- Give the tools to an agent with **`tools=mcp_tools`**.
- Install support once with `pip install "crewai-tools[mcp]"`.
- Run the MCP demo as a **`.py` file** (`python mcp_agent_demo.py`) — see `mcp_math_server.py` and `mcp_agent_demo.py`.